# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen/"
temp_files = originals + "temp/"
complete_files = originals + "complete/"

# Day we're updating data
update_date = "05-14-2025"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# print(len(metadata)) # 7397 rows

# Get rid of missing dates; they won't be counted anyway
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]

# # Find only >= 2024 to start
# metadata["Collection_Date_Compare"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata = metadata[metadata["Collection_Date_Compare"] >= datetime(2021, 11, 1).strftime("%Y-%m-%d")] # Note that those with only years will default to today

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= datetime(2025, 4, 14).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= datetime(2025, 5, 14).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025

571


In [3]:
# Get list of genotypes

# os.chdir(home)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["B3.13"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

metadata["Genotype"] = genoflu_results["Genotype"]
# metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes
metadata = metadata[metadata["Genotype"] == genotypes[0]]

# Get only the genotypes we want: B3.13 and D1.1

# b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

# metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

153


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,Genotype
8388,SRR33125012,WGS,148.14,103837402,PRJNA1207547,SAMN47941494,Viral,38567600,USDA-NVSL,2025,...,2025-04-14 15:01:46,1,25-004426-001,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS24712538,False,NaN,B3.13
8389,SRR33125013,WGS,148.47,151899011,PRJNA1207547,SAMN47941493,Viral,55900173,USDA-NVSL,2025,...,2025-04-14 14:59:12,1,25-004499-002,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS24712537,False,NaN,B3.13
8390,SRR33125014,WGS,147.45,166656358,PRJNA1207547,SAMN47941492,Viral,60793010,USDA-NVSL,2025,...,2025-04-14 14:59:27,1,25-004497-005,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS24712536,False,NaN,B3.13
8391,SRR33125015,WGS,148.33,107664413,PRJNA1207547,SAMN47941491,Viral,39666477,USDA-NVSL,2025,...,2025-04-14 15:00:01,1,25-004497-004,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS24712534,False,NaN,B3.13
8392,SRR33125016,WGS,120.38,2476776,PRJNA1207547,SAMN47941490,Viral,986034,USDA-NVSL,2025,...,2025-04-14 14:59:20,1,25-004480-001,SRP557452,NaN,tracheal swab,SRS24712535,False,NaN,B3.13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8862,SRR33370187,WGS,149.06,266567663,PRJNA1207547,SAMN48201686,Viral,96596674,USDA-NVSL,2025,...,2025-04-29 13:43:32,1,25-012740-001,SRP557452,NaN,CLOACAL/TRACHEAL SWAB POOL,SRS24885803,False,NaN,B3.13
8863,SRR33370188,WGS,149.12,252084447,PRJNA1207547,SAMN48201685,Viral,91406525,USDA-NVSL,2025,...,2025-04-29 13:43:40,1,25-012695-001,SRP557452,NaN,BLOOD SWAB,SRS24885802,False,NaN,B3.13
8864,SRR33370189,WGS,147.86,126955445,PRJNA1207547,SAMN48201684,Viral,52294003,USDA-NVSL,2025,...,2025-04-29 13:43:27,1,25-012058-002,SRP557452,NaN,CLOACAL/TRACHEAL SWAB POOL,SRS24885801,False,NaN,B3.13
8865,SRR33370190,WGS,148.58,124032966,PRJNA1207547,SAMN48201654,Viral,50995262,USDA-NVSL,2025,...,2025-04-29 13:43:26,1,25-012058-001,SRP557452,NaN,CLOACAL/TRACHEAL SWAB POOL,SRS24885800,False,NaN,B3.13


In [5]:
# # Get specific geolocation from genbank_mapping.tsv

# genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
# genbank_mapping["Run"] = genbank_mapping["sra_run"]
# genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
# genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# print(genbank_mapping["name_state"])
# print(len(metadata_genbank))
# display(metadata_genbank) # Maybe there is no state information since 3/18/2025?

In [6]:
# If no states

metadata_genbank = metadata

metadata_genbank["name_state"] = "USA"

## Get and save collection date

In [7]:

# # Get all dates
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))

# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank.csv")

In [8]:
# # Upload saved data -- if doing this, make sure the above cell is commented out
# os.chdir(temp_files + "saved/")
# metadata_genbank = pd.read_csv("metadata_genbank_4-18-2025.csv")
# os.chdir(temp_files)

# # Get only updated dates

# unknown_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] == "2024") | (metadata_genbank["Collection_Date_Specific"] == "2025")] # Dates we don't have
# known_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] != "2024") & (metadata_genbank["Collection_Date_Specific"] != "2025")] # Dates we've already gotten

# # Get new dates also 
# # new_dates = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

# updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible

# metadata_genbank = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Remove pre-2024 dates

# metadata_genbank["Collection_Date_Compare"] = metadata_genbank["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata_genbank = metadata_genbank[metadata_genbank["Collection_Date_Compare"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

# print(metadata_genbank[["Collection_Date_Specific"]])

# display(metadata_genbank)

In [9]:
# If no collection dates

metadata_genbank["Collection_Date_Specific"] = metadata_genbank["Collection_Date"]

## Get host type

In [10]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genbank)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['dunlin', 'snow goose', 'hooded merganser', 'rough-legged hawk', 'bufflehead', 'cattle', 'red-tailed hawk', 'red fox', 'herring gull', 'crow', 'trumpeter swan', 'duck', 'black vulture', 'osprey', "bonaparte's gull", 'blue-winged teal', 'flamingo', 'green-winged teal', 'bald eagle', 'cat', 'red-breasted merganser', 'great horned owl', 'lesser scaup', 'snowy owl', 'american crow', 'american black duck', 'mallard', 'vulture', 'turkey vulture', 'pet food', 'canada goose', 'sand crane', "cooper's hawk", 'wood duck']
[]
                 avian               cattle        feline   other_mammal  \
0     great_horned_owl            dairy_cow           cat     deer mouse   
1         common_raven               cattle  domestic_cat    house_mouse   
2        cooper's_hawk  cattle milk product     feral_cat          skunk   
3         coopers_hawk          bovine_milk        feline  striped_skunk   
4              peafowl              bovine   domestic-cat     norway rat   
..                 ... 

In [11]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_genbank, animals_ref) # Get host type

metadata_genbank["years"] = metadata_genbank["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [30]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date_Specific"]):
    
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date_Specific"] = metadata_genbank.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata_genbank.loc[num, "Collection_Date_Specific"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata_genbank.loc[num, "Collection_Date_Specific"] = date

    metadata_genbank = metadata_genbank.dropna(thresh=2)

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"].apply(lambda x: str(x)) + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

display(metadata_genbank[["ReleaseDate", 'create_date']])

,ReleaseDate,create_date
8388,2025-04-14,2025-04-14 15:01:46
8389,2025-04-14,2025-04-14 14:59:12
8390,2025-04-14,2025-04-14 14:59:27
8391,2025-04-14,2025-04-14 15:00:01
8392,2025-04-14,2025-04-14 14:59:20
...,...,...
8862,2025-05-02,2025-04-29 13:43:32
8863,2025-05-02,2025-04-29 13:43:40
8864,2025-05-02,2025-04-29 13:43:27
8865,2025-05-02,2025-04-29 13:43:26


In [13]:
print(metadata_genbank)

              Run Assay Type  AvgSpotLen        Bases    BioProject  \
8388  SRR33125012        WGS      148.14  103837402.0  PRJNA1207547   
8389  SRR33125013        WGS      148.47  151899011.0  PRJNA1207547   
8390  SRR33125014        WGS      147.45  166656358.0  PRJNA1207547   
8391  SRR33125015        WGS      148.33  107664413.0  PRJNA1207547   
8392  SRR33125016        WGS      120.38    2476776.0  PRJNA1207547   
...           ...        ...         ...          ...           ...   
8862  SRR33370187        WGS      149.06  266567663.0  PRJNA1207547   
8863  SRR33370188        WGS      149.12  252084447.0  PRJNA1207547   
8864  SRR33370189        WGS      147.86  126955445.0  PRJNA1207547   
8865  SRR33370190        WGS      148.58  124032966.0  PRJNA1207547   
8866  SRR33370191        WGS      148.75  261906821.0  PRJNA1207547   

         BioSample BioSampleModel       Bytes Center Name Collection_Date  \
8388  SAMN47941494          Viral  38567600.0   USDA-NVSL            2

## Make FASTA files

In [14]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [15]:
# print(fasta_files.keys())

In [16]:
# Create fasta files 

os.chdir(originals + "complete/B3_13_D1_1/04-14-2025--05-14-2025_D1_1/")
names = []
for pair in fasta_files.keys():
    output_path = originals + "complete/B3_13_D1_1/04-14-2025--05-14-2025_D1_1/" + pair + "_andersen_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names)/8)

>A/FLAMINGO/USA/25-004426-001/2025|H5N1|2025|avian|B3.13
>A/DUNLIN/USA/25-004499-002/2025|H5N1|2025|avian|B3.13
>A/DUNLIN/USA/25-004497-005/2025|H5N1|2025|avian|B3.13
>A/DUNLIN/USA/25-004497-004/2025|H5N1|2025|avian|B3.13
>A/DUCK/USA/25-004480-001/2025|H5N1|2025|avian|B3.13
>A/DUCK/USA/25-004477-001/2025|H5N1|2025|avian|B3.13
>A/CAT/USA/25-006823-001/2025|H5N1|2025|feline|B3.13
>A/CAT/USA/25-006491-001/2025|H5N1|2025|feline|B3.13
>A/BALD EAGLE/USA/25-005869-001/2025|H5N1|2025|avian|B3.13
>A/CAT/USA/25-006047-001/2025|H5N1|2025|feline|B3.13
>A/CROW/USA/25-004431-001/2025|H5N1|2025|avian|B3.13
>A/CANADA GOOSE/USA/25-004425-001/2025|H5N1|2025|avian|B3.13
>A/CANADA GOOSE/USA/25-004417-001/2025|H5N1|2025|avian|B3.13
>A/CANADA GOOSE/USA/25-004415-016/2025|H5N1|2025|avian|B3.13
>A/CANADA GOOSE/USA/25-004415-012/2025|H5N1|2025|avian|B3.13
>A/CANADA GOOSE/USA/25-004415-011/2025|H5N1|2025|avian|B3.13
>A/CANADA GOOSE/USA/25-004415-006/2025|H5N1|2025|avian|B3.13
>A/CANADA GOOSE/USA/25-004415-005/2

## De-Duplication

In [17]:
# De-duplication 

# Gisaid 

gisaid = downloads + "GISAID/complete/B3_13_D1_1/04-14-2025--05-14-2025_B3_13_northa/"

os.chdir(gisaid)

dfs_gisaid = create_dataframes(gisaid)
# dfs_gisaid2 = create_dataframes(gisaid2)

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2


In [18]:
# Do the same with Andersen 

dfs_andersen = create_dataframes(originals + "complete/B3_13_D1_1/04-14-2025--05-14-2025_B3_13/")

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2


In [19]:
# os.chdir(downloads)
# dfs_gisaid["B3.13_HA"].to_csv

In [20]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
defaultdict(<class 'list'>, {'B3.13_HA': [    isolate_partial                                        full_header  \
0        004426-001  >A/FLAMINGO/USA/25-004426-001/2025|H5N1|2025|a...   
1        004499-002  >A/DUNLIN/USA/25-004499-002/2025|H5N1|2025|avi...   
2        004497-005  >A/DUNLIN/USA/25-004497-005/2025|H5N1|2025|avi...   
3        004497-004  >A/DUNLIN/USA/25-004497-004/2025|H5N1|2025|avi...   
4        004480-001  >A/DUCK/USA/25-004480-001/2025|H5N1|2025|avian...   
..              ...                                                ...   
148      012740-001  >A/RED-TAILED HAWK/USA/25-012740-001/2025|H5N1...   
149      012695-001  >A/RED-TAILED HAWK/USA/25-012695-001/2025|H5N1...   
150      012058-002  >A/RED-TAILED HAWK/USA/25-012058-002/2025|H5N1...   
151      012058-001  >A/HERRING GULL/USA/25-012058-001/2025|H5N1|20...   
152      012269-006  >A/AMERICAN BLACK DUCK/USA/25-012269-006/2025|...

In [21]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

8
8


In [27]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
# same = []
# andersen = set()
# gisaid = set()

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
                # same.append(gisaid_key)
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                print(len(andersen_df))
                # print(andersen_df)
                gisaid_df = dfs_gisaid[gisaid_key][0]
                print(len(gisaid_df))
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                print(pd.concat([gisaid_df, andersen_df]).drop_duplicates())

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                print("len full df:", len(full_df))
                test = len(full_df.drop_duplicates(subset="isolate_partial"))

                dedup_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")

                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print(full_df)
                
                print("Keeping nothing: ", test)
                
                print("len deduplicated:", len(dedup_df))
                full_dfs[andersen_key].append(dedup_df)
            # else:
                # gisaid.add(gisaid_key)
                # andersen.add(andersen_key)
    
    # break 


# print(full_dfs)
# print(len(full_dfs))
# print(319*8)
# print(len(same))
# print(len(andersen))
# print(len(gisaid))

153
184
          isolate_partial                                        full_header  \
0    Inoc-TX-24-029328-01  >A/dairy_cow/Kansas/Inoc-TX-24-029328-01/2024|...   
1              012903-002  >A/dairy_cow/USA/012903-002/2025|H5N1|2025|cat...   
2              012903-003  >A/dairy_cow/USA/012903-003/2025|H5N1|2025|cat...   
3              012902-001  >A/dairy_cow/USA/012902-001/2025|H5N1|2025|cat...   
4              012902-002  >A/dairy_cow/USA/012902-002/2025|H5N1|2025|cat...   
..                    ...                                                ...   
148            012740-001  >A/RED-TAILED HAWK/USA/25-012740-001/2025|H5N1...   
149            012695-001  >A/RED-TAILED HAWK/USA/25-012695-001/2025|H5N1...   
150            012058-002  >A/RED-TAILED HAWK/USA/25-012058-002/2025|H5N1...   
151            012058-001  >A/HERRING GULL/USA/25-012058-001/2025|H5N1|20...   
152            012269-006  >A/AMERICAN BLACK DUCK/USA/25-012269-006/2025|...   

                               

In [23]:
# If none in one database, only use the other and drop duplicates

# full_dfs = defaultdict(list)
# for key in dfs_andersen.keys():
#     print(key)
# # for key in ["D1.3"]:
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             # full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)

## Create FASTA files combining Andersen and GISAID

In [24]:
# Create FASTA files per segment

combined_files = downloads + "Combinations/GISAID_Andersen/B3_13_D1_1/04-14-2025--05-14-2025_B3_13/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
